---

## ModelFallbackMiddleware 테스트

``ModelFallbackMiddleware`` 는 **primary 모델** 호출이 실패하면
등록된 **폴백 모델**을 순서대로 시도합니다.

- 훅: ``wrap_model_call`` — ``handler(request)`` 실패 시 ``request.override(model=...)`` 로 교체

**참고:** [Built-in Middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in)

아래 각 섹션은 ``feature/MiddlewareModelFallback.py`` 의 ``MiddlewareModelFallbackAgent`` 로
동작을 확인합니다.

> **API 키:** primary·폴백 모델 제공자에 맞는 키가 필요합니다 (OpenAI, Anthropic 등).

**구성 요약:**

| 위치 | 역할 |
|:---|:---|
| ``create_agent(model=...)`` | 1차(primary) 모델 — 먼저 호출 |
| ``ModelFallbackMiddleware(first, *rest)`` | primary 실패 시 **순서대로** 시도할 폴백 |
| ``fallback_models`` (생성자) | 폴백 목록을 튜플로 넘김 |

**실행 흐름**

```
primary 호출 → 성공 → 끝
           → 실패 → fallback[0] → 성공 → 끝
                          → 실패 → fallback[1] → …
                                              → 모두 실패 → 마지막 예외 re-raise
```

기본 폴백 체인(``_DEFAULT_FALLBACK_MODELS``): ``claude-haiku-4-5`` → ``openai:gpt-4.1-mini``

In [2]:
from langchain_core.messages import HumanMessage

from feature.MiddlewareModelFallback import (
    MiddlewareModelFallbackAgent,
    make_default_fallback_middleware,
    make_model_fallback_middleware,
)
from util.chat_model_enums import LangChainChatModel


### 1. primary 성공 — 폴백 미사용

유효한 primary 모델이면 폴백 없이 **정상 응답**이 나와야 합니다.

In [3]:
agent_ok = MiddlewareModelFallbackAgent()

result = agent_ok.invoke(
    inputs={"messages": [HumanMessage(content="Say hello in one short sentence.")]},
)

assert result is not None
print("✓ primary 성공 — 응답:", result["messages"][-1].content[:120])


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

Hello! How can I help you today?
✓ primary 성공 — 응답: Hello! How can I help you today?
